In [1]:
import os
import cv2
import glob
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm
import warnings

# 过滤警告
warnings.filterwarnings('ignore')

# ==========================================
# 1. 核心配置 (针对 Kaggle P100 优化)
# ==========================================
class CFG:
    seed = 42
    # 训练时裁剪的大小
    train_img_size = 512   
    # 验证时保持原图大小 (P100推理1024没问题)
    val_img_size = 1024    
    
    # 训练Batch Size: 512尺寸下，P100可以开到 8-16
    train_batch_size = 8   
    # 验证Batch Size: 1024尺寸较大，建议设小一点，防止峰值显存溢出
    val_batch_size = 4     
    
    epochs = 150           
    lr = 2e-4             
    min_lr = 1e-6         
    weight_decay = 0.05   
    num_workers = 2        # Kaggle环境建议设为2或0，太高容易报错
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    base_path = "/kaggle/input/levir-cd/LEVIR CD" # 请确保路径正确
    model_path = "TransConv_SOTA_Best.pth"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 

seed_everything(CFG.seed)

# ==========================================
# 2. 数据集 (分块训练，全图验证)
# ==========================================
class LEVIR_Dataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.mode = mode
        self.t1_paths = sorted(glob.glob(os.path.join(root_dir, mode, 'A', '*.png')))
        self.t2_paths = sorted(glob.glob(os.path.join(root_dir, mode, 'B', '*.png')))
        self.label_paths = sorted(glob.glob(os.path.join(root_dir, mode, 'label', '*.png')))
        
        if mode == 'train':
            # 训练集：强数据增强 + 随机裁剪 (保持原分辨率细节)
            self.aug = A.Compose([
                # 随机裁剪 512x512，模型每次看图的一个局部
                A.RandomCrop(height=CFG.train_img_size, width=CFG.train_img_size, p=1.0),
                
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Transpose(p=0.5),
                
                # 几何与颜色变换
                A.OneOf([
                    A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
                    A.GridDistortion(p=0.5),
                    A.OpticalDistortion(distort_limit=1, shift_limit=0.5, p=0.5),
                ], p=0.3),
                A.OneOf([
                    A.GaussNoise(p=0.5),
                    A.RandomBrightnessContrast(p=0.5),
                    A.HueSaturationValue(p=0.5),
                ], p=0.3),
                
                A.Normalize(),
                ToTensorV2()
            ], additional_targets={'image_0': 'image'})
        else:
            # 验证集：只做归一化，保留 1024x1024 全图
            # P100 16G 足够推理全图，这样精度最高
            self.aug = A.Compose([
                A.Normalize(),
                ToTensorV2()
            ], additional_targets={'image_0': 'image'})

    def __len__(self): return len(self.t1_paths)

    def __getitem__(self, idx):
        t1 = cv2.imread(self.t1_paths[idx]); t1 = cv2.cvtColor(t1, cv2.COLOR_BGR2RGB)
        t2 = cv2.imread(self.t2_paths[idx]); t2 = cv2.cvtColor(t2, cv2.COLOR_BGR2RGB)
        label = cv2.imread(self.label_paths[idx], cv2.IMREAD_GRAYSCALE)
        label = (label > 128).astype(np.float32)

        augmented = self.aug(image=t1, image_0=t2, mask=label)
        return augmented['image'], augmented['image_0'], augmented['mask'].unsqueeze(0)

# ==========================================
# 3. 核心模块：Transformer & CNN 混合
# ==========================================
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm_q = nn.LayerNorm(dim)
        self.norm_k = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x1, x2):
        B, C, H, W = x1.shape
        # 这里 flatten 会自动适应 H, W，所以无论是训练的 16x16 还是验证的 32x32 都能跑
        x1_flat = x1.flatten(2).transpose(1, 2)
        x2_flat = x2.flatten(2).transpose(1, 2)
        
        q1 = self.norm_q(x1_flat); k2 = self.norm_k(x2_flat)
        attn_out1, _ = self.attn(q1, k2, k2)
        x1_out = x1_flat + attn_out1
        x1_out = x1_out + self.ffn(x1_out)
        
        q2 = self.norm_q(x2_flat); k1 = self.norm_k(x1_flat)
        attn_out2, _ = self.attn(q2, k1, k1)
        x2_out = x2_flat + attn_out2
        x2_out = x2_out + self.ffn(x2_out)

        return x1_out.transpose(1, 2).reshape(B, C, H, W), x2_out.transpose(1, 2).reshape(B, C, H, W)

class SpatialDifferenceModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 1)
        )
    def forward(self, t1, t2):
        return self.conv(torch.cat([t1, t2], dim=1))

# ==========================================
# 4. 整体架构
# ==========================================
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_ch, out_ch // 8, 1),
            nn.ReLU(inplace=True), nn.Conv2d(out_ch // 8, out_ch, 1), nn.Sigmoid()
        )
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        if skip is not None: x = torch.cat([x, skip], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x * self.se(x)

class TransConvNeXt_CD(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('convnext_tiny', pretrained=True, features_only=True)
        # Interaction
        self.inter_1 = SpatialDifferenceModule(96)
        self.inter_2 = SpatialDifferenceModule(192)
        self.inter_3 = SpatialDifferenceModule(384)
        self.transformer_inter = CrossAttentionBlock(dim=768, num_heads=12)
        self.inter_4_fuse = nn.Conv2d(768 * 2, 768, 1)
        # Decoder
        self.dec4 = DecoderBlock(768, 384, 384)
        self.dec3 = DecoderBlock(384, 192, 192)
        self.dec2 = DecoderBlock(192, 96, 96)
        self.dec1 = DecoderBlock(96, 0, 64)
        self.final_conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )
        self.head_d4 = nn.Conv2d(384, 1, 1)
        self.head_d3 = nn.Conv2d(192, 1, 1)

    def forward(self, t1, t2):
        f1 = self.encoder(t1); f2 = self.encoder(t2)
        
        diff_1 = self.inter_1(f1[0], f2[0])
        diff_2 = self.inter_2(f1[1], f2[1])
        diff_3 = self.inter_3(f1[2], f2[2])
        
        t1_out, t2_out = self.transformer_inter(f1[3], f2[3])
        diff_4 = self.inter_4_fuse(torch.cat([t1_out, t2_out], dim=1))
        
        d4 = self.dec4(diff_4, diff_3)
        d3 = self.dec3(d4, diff_2)
        d2 = self.dec2(d3, diff_1)
        d1 = self.dec1(d2)
        out = self.final_conv(d1)
        
        if self.training:
            return out, self.head_d4(d4), self.head_d3(d3)
        return out

# ==========================================
# 5. 损失与指标
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        
    def dice_loss(self, pred, target):
        probs = torch.sigmoid(pred)
        intersection = (probs * target).sum()
        union = probs.sum() + target.sum()
        return 1 - (2. * intersection + 1e-6) / (union + 1e-6)

    def forward(self, preds, target):
        if isinstance(preds, tuple):
            final, d4, d3 = preds
            # 动态插值，自适应输入大小 (512 或 1024 都能对齐)
            t_d3 = F.interpolate(target, size=d3.shape[2:], mode='nearest')
            t_d4 = F.interpolate(target, size=d4.shape[2:], mode='nearest')
            
            l_main = self.bce(final, target) + self.dice_loss(final, target)
            l_d3 = self.bce(d3, t_d3) + self.dice_loss(d3, t_d3)
            l_d4 = self.bce(d4, t_d4) + self.dice_loss(d4, t_d4)
            return l_main + 0.4 * l_d3 + 0.2 * l_d4
        else:
            return self.bce(preds, target) + self.dice_loss(preds, target)

def predict_tta(model, t1, t2):
    # 全图 TTA (1024x1024)
    model.eval()
    out = torch.sigmoid(model(t1, t2))
    out += torch.flip(torch.sigmoid(model(torch.flip(t1, [3]), torch.flip(t2, [3]))), [3])
    out += torch.flip(torch.sigmoid(model(torch.flip(t1, [2]), torch.flip(t2, [2]))), [2])
    return out / 3.0

class Metrics:
    def __init__(self): self.reset()
    def reset(self): self.tp, self.fp, self.fn = 0, 0, 0
    def update(self, pred, target):
        pred = (pred > 0.5).float()
        self.tp += (pred * target).sum().item()
        self.fp += (pred * (1 - target)).sum().item()
        self.fn += ((1 - pred) * target).sum().item()
    def get(self):
        f1 = 2*self.tp / (2*self.tp + self.fp + self.fn + 1e-8)
        iou = self.tp / (self.tp + self.fp + self.fn + 1e-8)
        p = self.tp / (self.tp + self.fp + 1e-8)
        r = self.tp / (self.tp + self.fn + 1e-8)
        return f1, iou, p, r

# ==========================================
# 6. 训练主流程
# ==========================================
def train_model():
    print(f"[Info] 显卡: {torch.cuda.get_device_name(0)}")
    print(f"[Info] 训练模式: Random Crop ({CFG.train_img_size}x{CFG.train_img_size})")
    print(f"[Info] 验证模式: Full Image ({CFG.val_img_size}x{CFG.val_img_size})")

    train_ds = LEVIR_Dataset(CFG.base_path, 'train')
    val_ds = LEVIR_Dataset(CFG.base_path, 'val')
    
    # 不同的 Batch Size 配置
    train_loader = DataLoader(train_ds, batch_size=CFG.train_batch_size, shuffle=True, 
                              num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.val_batch_size, shuffle=False, 
                            num_workers=CFG.num_workers, pin_memory=True)
    
    model = TransConvNeXt_CD().to(CFG.device)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=CFG.min_lr)
    criterion = HybridLoss()
    scaler = torch.cuda.amp.GradScaler()
    met = Metrics()
    
    best_f1 = 0.0
    print(">>> 🚀 开始训练 TransConvNeXt (Patch Train / Full Val)...")
    
    for epoch in range(CFG.epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.epochs}")
        
        for t1, t2, mask in pbar:
            t1, t2, mask = t1.to(CFG.device), t2.to(CFG.device), mask.to(CFG.device)
            
            optimizer.zero_grad()
            # 混合精度训练，节省显存
            with torch.cuda.amp.autocast():
                preds = model(t1, t2)
                loss = criterion(preds, mask)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=optimizer.param_groups[0]['lr'])
            
        scheduler.step()
        
        # Validation (在 1024 大图上进行)
        met.reset()
        torch.cuda.empty_cache() # 清理训练时的缓存，给验证腾出空间
        with torch.no_grad():
            for t1, t2, mask in val_loader:
                t1, t2, mask = t1.to(CFG.device), t2.to(CFG.device), mask.to(CFG.device)
                pred = predict_tta(model, t1, t2)
                met.update(pred, mask)
        
        f1, iou, p, r = met.get()
        print(f"[*] Epoch {epoch+1} | F1: {f1:.4f} | IoU: {iou:.4f} | Prec: {p:.4f} | Rec: {r:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), CFG.model_path)
            print(f">>> 👑 SOTA Update! Best F1: {best_f1:.4f}")

if __name__ == '__main__':
    if os.path.exists(CFG.base_path):
        train_model()
    else:
        print("请检查数据集路径！")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

[Info] 显卡: Tesla P100-PCIE-16GB
[Info] 训练模式: Random Crop (512x512)
[Info] 验证模式: Full Image (1024x1024)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

>>> 🚀 开始训练 TransConvNeXt (Patch Train / Full Val)...


Epoch 1/150:   0%|          | 0/55 [00:00<?, ?it/s]

...

[*] Epoch 148 | F1: 0.8973 | IoU: 0.8138 | Prec: 0.9026 | Rec: 0.8921


Epoch 149/150:   0%|          | 0/55 [00:00<?, ?it/s]

[*] Epoch 149 | F1: 0.9022 | IoU: 0.8219 | Prec: 0.8945 | Rec: 0.9101


Epoch 150/150:   0%|          | 0/55 [00:00<?, ?it/s]

[*] Epoch 150 | F1: 0.9035 | IoU: 0.8239 | Prec: 0.9036 | Rec: 0.9033
